In [1]:
import pandas as pd
from pandas.api.types import is_numeric_dtype

In [2]:
merged_all = pd.read_parquet("/Users/yutung/MQF/Machine learning/Project/ml_project/data/merged_all.parquet")
df = merged_all.copy()

In [3]:
# rename columns for clarity
rename_map = {
    "yield":      "ytm",               # bond yield to maturity
    "ret":        "stock_ret",         # stock return
    "ret_eom":    "bond_ret",          # bond end-of-month return
    "vol":        "stock_vol",         # stock volatility
    "mktcap":     "stock_mktcap",      # from daily stock
    "mkt_cap":    "accounting_mktcap", # from fundamentals
    "price_mean": "stock_price_mean",  # avg stock price
}
df = df.rename(columns={k: v for k, v in rename_map.items() if k in df.columns})

In [4]:
# rename the main ID to something clear
df = df.rename(columns={"cusip_x": "bond_cusip"})

# drop all duplicate / merge-only identifiers
drop_cols = [
    "company_symbol",
    "cusip_y",
    "GVKEY",
    "PERMNO",
    "issuer6",
    "sector_etf",
    "gsector",
    "month_x",
    "month_y"
]

df = df.drop(columns=drop_cols, errors="ignore")

## Define target and build X,Y

In [5]:
target_col = "ytm"
id_col = "bond_cusip"
date_col = "date"

# drop missing target
df = df[~df[target_col].isna()].copy()

# y
y = df[[date_col, id_col, target_col]].copy()

# x (keep date + id + all other features)
feature_cols = [c for c in df.columns if c not in [target_col]]
x = df[feature_cols].copy()

# ensure date is datetime
x[date_col] = pd.to_datetime(x[date_col])

## Detect categoricals

In [6]:
# convert costat to numeric (A / I -> 1 / 0, others -> NaN)
costat_map = {
    "A": 1,   # active
    "I": 0,   # inactive
}
df["costat_bin"] = df["costat"].map(costat_map)
df.drop(columns=["costat"], errors="ignore", inplace=True)

# collect all binary numeric columns (including ratings, upgrade/downgrade, costat_bin, etc.)
binary_cols = []

for col in df.columns:
    if is_numeric_dtype(df[col]):
        n = df[col].nunique(dropna=True)
        if n == 2:  # truly binary
            binary_cols.append(col)

print("\nBinary numeric columns:")
for col in binary_cols:
    print(col, df[col].unique())


Binary numeric columns:
rating_A [0. 1.]
rating_AA [0. 1.]
rating_AAA [0. 1.]
rating_B [0. 1.]
rating_BB [0. 1.]
rating_BBB [0. 1.]
rating_C [0. 1.]
rating_CC [0. 1.]
rating_CCC [0. 1.]
rating_D [0. 1.]
upgrade [0. 1.]
downgrade [0. 1.]
costat_bin [ 1.  0. nan]


In [7]:
numeric_cols = [
    col for col in df.columns
    if (is_numeric_dtype(df[col]) 
        and col not in binary_cols 
        and col not in id_col
        and col != target_col)
]

## Fill NaN with median

In [8]:
df["month"] = df["date"].dt.to_period("M")
for col in numeric_cols:
    df[col] = df.groupby("month")[col].transform(lambda x: x.fillna(x.median()))

for col in binary_cols:
    df[col] = df.groupby("month")[col].transform(lambda x: x.fillna(x.median()))

/opt/homebrew/Caskroom/miniconda/base/envs/ml/lib/python3.12/site-packages/numpy/lib/nanfunctions.py:1215: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)
/opt/homebrew/Caskroom/miniconda/base/envs/ml/lib/python3.12/site-packages/numpy/lib/nanfunctions.py:1215: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)
/opt/homebrew/Caskroom/miniconda/base/envs/ml/lib/python3.12/site-packages/numpy/lib/nanfunctions.py:1215: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)
/opt/homebrew/Caskroom/miniconda/base/envs/ml/lib/python3.12/site-packages/numpy/lib/nanfunctions.py:1215: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)


In [9]:
# check NaN 
nan_table = (
    df.isna()
      .mean()
      .sort_values(ascending=False)
      .to_frame("nan_pct")
)

nan_table["nan_count"] = df.isna().sum()

print(nan_table.head(5))


                   nan_pct  nan_count
t_spread          0.000133        115
date              0.000000          0
revtq_growth      0.000000          0
stock_price_mean  0.000000          0
log_atq           0.000000          0


In [10]:
# fill NaN with 0
df["t_spread"] = df["t_spread"].fillna(0)

## Normalize

In [11]:
for col in binary_cols:
    df[col] = df[col].map({0: -1, 1: 1})

In [12]:
# for macro features, keep raw values for these columns
macro_keep_raw = [
    "sp500_ret", "ir3m_chg", "ir10y_chg",
    "vix_chg", "gdp_gr", "cpi_infl", "gs3m", "term_spread"
]

numeric_cols_to_rank = [col for col in numeric_cols if col not in macro_keep_raw]

def rank_normalize(s):
    r = s.rank(method="average")
    r = (r - r.min()) / (r.max() - r.min() + 1e-9)
    return r * 2 - 1

# apply rank-normalization only on the right columns
for col in numeric_cols_to_rank:
    df[col] = df.groupby("month")[col].transform(rank_normalize)

In [13]:
X_cols = binary_cols + numeric_cols
final_df = df[[id_col] + [target_col] +[date_col] + X_cols].copy()

print(final_df.head())
print(final_df.shape)

  bond_cusip      ytm       date  rating_A  rating_AA  rating_AAA  rating_B  \
1  000361AB1  0.04827 2002-08-31        -1         -1          -1        -1   
2  000361AB1  0.04386 2002-09-30        -1         -1          -1        -1   
3  000361AB1  0.04122 2002-10-31        -1         -1          -1        -1   
4  000361AB1  0.03873 2002-11-30        -1         -1          -1        -1   
5  000361AB1  0.04015 2002-12-31        -1         -1          -1        -1   

   rating_BB  rating_BBB  rating_C  ...  revtq_growth  niq_growth  sp500_ret  \
1         -1          -1        -1  ...     -0.376664    0.382235  -0.079004   
2         -1          -1        -1  ...     -0.293506    0.738528   0.004881   
3         -1          -1        -1  ...      0.427600    0.852125  -0.110024   
4         -1          -1        -1  ...      0.444784    0.839266   0.086449   
5         -1          -1        -1  ...      0.276607   -0.706236   0.057070   

   ir3m_chg  ir10y_chg   vix_chg    gdp_gr  

In [14]:
output_path = "/Users/yutung/MQF/Machine learning/Project/ml_project/data/final_merged_clean.parquet"

final_df.to_parquet(output_path, index=True)
